# Lecture 21: Solving the Diffusion Partial Differential Equation

In addition to things that are conserved, one often needs to model a partial differential equation where some physical effect causes energy to enter or leave the system. Such systems are often modelled with diffusion equations. This lecture discusses how to solve such equations with implicit partial differential equation solvers.

In addition to the lecture material below, all students also need to download `lec21_module.py` using the command below:
```bash
curl -O https://raw.githubusercontent.com/PsuAstro410/psuastro410.github.io/refs/heads/main/lectures/lec21_module.py
```

### Accepting and submitting

To accept this assignment on Classroom 50 with your GitHub profile enrolled in the class, click on the link \
https://classroom50.org/PsuAstro410/410astro26/assignments/lecture-21/accept \
and follow the steps the webpage prompts.

To submit this assignment, follow the steps in "First time Classroom 50 setup" and "Accepting and Submitting Assignments" on the course webpage:\
https://psuastro410.github.io/tips/github/

### Resources and Acknowledgements

We have used material from the following sources: \
https://zingale.github.io/computational_astrophysics/diffusion/diffusion.html

We also thank Shane Davis for some material that went into today's lecture.

In [ ]:
# Nice things to have this lecture
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import lec21_module as pdemod
from scipy import linalg

# Diffusion

The [diffusion equation](https://en.wikipedia.org/wiki/Diffusion_equation) is

$$\phi_t = k \phi_{xx}$$

In astrophysics, this can describe thermal diffusion, where $\phi$
is temperature, diffusion of photons, where $\phi$ is the photon
energy density, mass diffusion, where $\phi$ is the concentration
of a particular species, or viscosity, where $\phi$ is velocity.

Physically, diffusion takes a strongly-peaked concentration of $\phi$
and spreads it out.  And initial discontinuities are quickly smeared out.
We can derive this via [Fick's law](https://en.wikipedia.org/wiki/Fick's_laws_of_diffusion)
which says that the diffusive flux is proportion to the gradient of the quantity 
diffusing:

$$F = -k \nabla \phi$$

where the "$-$" sign indicates that diffusion flows from high concentration
to low concentration.


**Note:**
More generally, the diffusion coefficient, $k$, can depend on
position (or even $\phi$), and we could write the diffusion equation
as:

$$\phi_t = \left [ -k \phi_x \right ]_x$$

We'll assume that $k$ is constant here.  In this form, this is also
called the
[heat equation](https://en.wikipedia.org/wiki/Heat_equation).

Solving the diffusion equation requires initial conditions and two
boundary conditions.


## (An) Analytic solution

Testing a numerical solution of the diffusion equation is easier if we
have an analytic solution.  A nice property of the diffusion equation
is that a Gaussian initial profile diffuses as a wider, lower amplitude
Gaussian:

$$\phi^a(x,t) = (\phi_2 - \phi_1) \left ( \frac{t_0}{t + t_0} \right )^{1/2}
e^{-\tfrac{1}{4} (x - x_c)^2/ k(t + t_0)} + \phi_1$$

Here, $t_0$ is a small time.  In the limit that $t_0 \rightarrow 0$,
we get a delta-function for the initial condition.

**Note:**
This is a solution of the 1D diffusion equation.  For 2 and 3D, the 
solution is slightly different.

If we look at the evolution of this Gaussian, we see:

![diffusion_solution](https://zingale.github.io/computational_astrophysics/_images/phi_analytic_t.png)

This shows the expected behavior&mdash;a strongly peaked concentration
of $\phi$ spreads out with time.

# Explicit Discretization of Diffusion

We'll start with our diffusion equation:

$$\phi_t = k \phi_{xx}$$

by replacing the time derivative with a forward-difference (first-order accurate)
and the second derivative in space with a centered difference (second-order accurate):

$$\frac{\phi_i^{n+1} - \phi_i^n}{\Delta t} = k \frac{\phi_{i+1}^n - 2 \phi_i^n + \phi_{i-1}^n}{\Delta x^2}$$

## Stability criterion

This is reminiscent of the FTCS method we saw with advection, so we need to ask&mdash;is this discretization stable?

As this equation is linear, we can do [von Neumann stability analysis](https://en.wikipedia.org/wiki/Von_Neumann_stability_analysis).  We look at a
single Fourier mode:

$$\phi_i^n = A^n e^{ji\theta}$$

where we are using $j = \sqrt{-1}$ for the imaginary unit.  Stability requires that:

$$\left | \frac{A^{n+1}}{A^n} \right | \le 1$$

When we work this out, we find that this discretization is stable as long as:

$$\Delta t \le \frac{1}{2} \frac{\Delta x^2}{k}$$

**Important:** Notice that the spatial dependence here is more severe than advection&mdash;we scale
as $\Delta x^2$.

We'll use a cell-centered finite-difference / finite-volume grid.

## Implementation

Included in `lec21_module.py` is the grid class.  It implements Neumann boundary conditions by simply setting the ghost cell values to the value just inside the domain:
```python
    def fill_BCs(self):
        """ fill the ghostcells with zero gradient (Neumann)
            boundary conditions """
        self.phi[0:self.ilo] = self.phi[self.ilo]
        self.phi[self.ihi+1:] = self.phi[self.ihi]
```

The other parts of the code are essentially identical to the grid that we used to solve the advection and burger's equation, except we solve for the variable `phi` rather than `a` or `u` now:
```python
        # storage for the solution
        self.phi = self.scratch_array()
```
and we also define a module for calculating the norm of some quantity that lives on the grid:
```python
    def norm(self, e):
        """ return the norm of quantity e which lives on the grid """
        if not len(e) == (2*self.ng + self.nx):
            return None

        return np.sqrt(self.dx*np.sum(e[self.ilo:self.ihi+1]**2))
```
You can look at `lec21_module.py` for the full code.

Here's the integration routine.  We require a function to be passed in with the initial conditions
of the form:

```python
init_cond(grid, k)
```

where `grid` is the `Grid` object and `k` is the thermal diffusivity.

We'll take a timestep based on the stability criterion:

$$\Delta t = \frac{C}{2} \frac{\Delta x^2}{k}$$

where $C < 1$ is a _safety factor_ that ensures that we remain stable.

In [ ]:
def diffuse_explicit(nx, k, C, tmax, init_cond):
    """ evolve the system with diffusion coeff k and CFL number C """

    ng = 1

    # create the grid
    g = pdemod.Grid(nx, ng)

    # time info
    dt = C * 0.5 * g.dx**2 / k
    t = 0.0

    # initialize the data
    g.phi[:] = init_cond(g, k)

    # evolution loop
    phinew = g.scratch_array()

    while t < tmax:

        # make sure we end right at tmax
        if t + dt > tmax:
            dt = tmax - t

        # fill the boundary conditions
        g.fill_BCs()

        alpha = k * dt / g.dx**2

        # loop over zones
        for i in range(g.ilo, g.ihi+1):

            # explicit diffusion
            phinew[i] = g.phi[i] + alpha * (g.phi[i+1] - 2.0*g.phi[i] + g.phi[i-1])

        # store the updated solution
        g.phi[:] = phinew[:]
        t += dt

    return g

Here are our initial conditions&mdash;the Gaussian profile.

In [ ]:
def gaussian_ic(g, k, t=0.0, t0=1.e-4, phi1=1.0, phi2=2.0):
    xc = 0.5*(g.xmin + g.xmax)
    return (phi2 - phi1) * (np.sqrt(t0/(t + t0)) *
                            np.exp(-0.25 * (g.x - xc)**2 / (k * (t + t0)))) + phi1

**Tip:** We can use this same function to compute the analytic solution at a time $t > 0$ by passing
the time in.

## A first run

We'll pick a maximum time in terms of the diffusion timescale,

$$\tau_\mathrm{diffuse} \sim \frac{\lambda^2}{k}$$

where $\lambda$ is the characteristic size we are diffusing over.  We'll
pick it as $\lambda \sim \Delta x$ since we start out very strongly peaked.

In [ ]:
nx = 64
C = 0.8

k = 1

t_diffuse = (1.0/nx)**2 / k

We'll run for a few different multiples of the characteristic time and
plot the evolution.  We'll also plot the analytic solution as a dotted line

In [ ]:
fig, ax = plt.subplots()

color = 0
for n in [0.01, 0.1, 1, 10]:
    g = diffuse_explicit(nx, k, C, n * t_diffuse, gaussian_ic)
    phi_analytic = gaussian_ic(g, k, t=n*t_diffuse)
    ax.plot(g.x[g.ilo:g.ihi+1], g.phi[g.ilo:g.ihi+1], label=f"t = {n * t_diffuse:7.5g}", c=f"C{color}")
    ax.plot(g.x[g.ilo:g.ihi+1], phi_analytic[g.ilo:g.ihi+1], ls=":", c=f"C{color}")
    color += 1

ax.set_xlim(0.35, 0.65)
ax.set_xlabel("$x$")
ax.set_ylabel(r"$\phi$")

ax.legend()
plt.show()

## Resolution

Note that it is critical to resolve the initial conditions.  Let's vary resolution to see what happens if we don't.  We'll plot the solution for different numbers of points all at the same time.

In [ ]:
fig, ax = plt.subplots()

for nx in [16, 32, 64, 128, 256]:
    g = diffuse_explicit(nx, k, C, 10 * t_diffuse, gaussian_ic)
    ax.plot(g.x[g.ilo:g.ihi+1], g.phi[g.ilo:g.ihi+1], label=f"nx = {nx}")

ax.legend()
ax.set_xlabel("$x$")
ax.set_ylabel(r"$\phi$")
plt.show()

## Exceeding stability

What happens if we exceed the timestep constraint?  Let's find out by setting C = 2.0

In [ ]:
fig, ax = plt.subplots()

nx - 64
C = 2.0

g = diffuse_explicit(nx, k, C, 10 * t_diffuse, gaussian_ic)
ax.plot(g.x[g.ilo:g.ihi+1], g.phi[g.ilo:g.ihi+1])

ax.set_xlabel("$x$")
ax.set_ylabel(r"$\phi$")
plt.show()

Check out that vertical scale! This is just like what we saw with advection.

## Convergence

How do we converge?

This is a bit trickier than our advection / elliptic cases, since now the truncation error is $\sim \mathcal{O}(\Delta t) + \mathcal{O}(\Delta x^2)$.  Note however, that our timestep is $\Delta t \sim \Delta x^2$, so we should still expect second order convergence as we increase the number of grid points.

In [ ]:
nx_reference = 64
t_diffuse = (1.0/nx_reference)**2 / k

In [ ]:
fig, ax = plt.subplots()

Ns = [16, 32, 64, 128, 256, 512]
dxs = []
errors = []

tmax = 10 * t_diffuse
C = 0.8

for nx in Ns:
    g = diffuse_explicit(nx, k, C, tmax, gaussian_ic)
    phi_analytic = gaussian_ic(g, k, t=tmax)
    dxs.append(g.dx)
    errors.append(g.norm(g.phi - phi_analytic))
    
ax.scatter(dxs, errors, marker="x")

ideal = errors[-1] * np.array(dxs)**2 / dxs[-1]**2

ax.plot(dxs, ideal, color="0.5", ls=":", label=r"$\mathcal{O}(\Delta x^2)$")

ax.legend()

ax.set_xscale("log")
ax.set_yscale("log")

ax.set_xlabel(r"$\Delta x$")
ax.set_ylabel("absolute error")
plt.show()

Here we see that we attain the expected $\mathcal{O}(\Delta x^2)$ convergence.  For coarse resolution, the fact that we are not really resolving the initial conditions leads to the large errors.

# Implicit Methods for Diffusion

One of the issues we have with our diffusion implementation is that the timestep constraint:

$$\Delta t = \frac{C}{2} \frac{\Delta x^2}{k}$$

can be very restrictive as $\Delta x$ gets small.  

**Note:**
In astrophysical simulations, we are often combining advection and diffusion,
and the diffusive timestep restriction can dominate, making our overall simulation
computationally expensive. For this reason, we often wish to use an _implicit-in-time_
integration method for diffusion.

Going back to our original discretization, we would write the implicit update as:

$$\frac{\phi_i^{n+1} - \phi_i^n}{\Delta t}
 = k \frac{\phi_{i+1}^{n+1} - 2 \phi_i^{n+1} + \phi_{i-1}^{n+1}}{\Delta x^2}$$
 
If we define

$$\alpha = \frac{k\Delta t}{\Delta x^2}$$

then we can write this as:

$$-\alpha \phi^{n+1}_{i-1} + (1 + 2\alpha) \phi_i^{n+1}
- \alpha \phi_{i+1}^{n+1} = \phi_i^n$$

This is a coupled set of algebraic equations for the new time state of $\phi$&mdash;a linear system.

**Important:**
The implicit discretization is stable for all values of $C$.  But keep in mind that
stability and accuracy are not the same thing.

There are several ways we can solve this:

* _direct solve_ : create a linear system of the form ${\bf A}{\bf x} = {\bf b}$
  and solve for the new state using Gaussian elimination or something similar
  
* _iterative solve_ : apply relaxation to the system, just like we did with our
  elliptic equations.
  

## Direct solve

In order to write down the linear system, we need to understand our boundary
conditions.  Consider the left boundary.  We'll denote the cell just inside
the domain as $\phi_\mathrm{lo}$ and the ghost cell just outside of the domain
as $\phi_{\mathrm{lo}-1}$.

* _Homeogeneous Neumann_ : for Neumann boundary conditions, we want the
  gradient at the boundary to be zero.  This means:
  
  $$\frac{\phi_\mathrm{lo} - \phi_{\mathrm{lo}-1}}{\Delta x} = 0$$
  
  This is second-order accurate at the physical boundary, $x_{\mathrm{lo}-1/2}$.
  
  Inserting this into our implicit system above, we find that the update for
  the first zone inside the boundary is:
  
  $$(1 + \alpha) \phi_\mathrm{lo}^{n+1} - \alpha \phi_{\mathrm{lo}+1}^{n+1} = \phi_\mathrm{lo}^n$$
  
* _Dirichlet_ : for Dirichlet boundary conditions, we want the value at the
  physical boundary to be specified&mdash;we'll call it $\phi_A$.  To second-order, that means that the
  average of the two zones on either side of the boundary should give us
  the desired value:
  
  $$\frac{\phi_\mathrm{lo} + \phi_{\mathrm{lo}-1}}{2} = \phi_A$$
  
  Inserting this into our implicit update, we find that the update for the first
  zone inside the boundary is:
  
  $$(1 + 3\alpha) \phi_\mathrm{lo}^{n+1} - \alpha \phi_{\mathrm{lo}+1}^{n+1} = \phi_\mathrm{lo}^n + 2 \alpha \phi_A$$
  
Similar constructions are used for the right boundary.

Putting this together, our linear system for implicit diffusion (assuming Neumann boundary conditions) appears as:

$$
\left (
\begin{array}{ccccccc}
1+\alpha &   -\alpha &           &        &         &           &          \\
-\alpha  & 1+2\alpha & -\alpha   &        &         &           &          \\
         & -\alpha   & 1+2\alpha & -\alpha&         &           &          \\
         &           & \ddots    & \ddots & \ddots  &           &          \\
         &           &           & \ddots & \ddots  & \ddots    &          \\
         &           &           &        & -\alpha & 1+2\alpha &-\alpha   \\
         &           &           &        &         & -\alpha   &1+\alpha \\
\end{array}
\right )
\left (
\begin{array}{c}
\phi_\mathrm{lo}^{n+1} \\
\phi_\mathrm{lo+1}^{n+1} \\
\phi_\mathrm{lo+2}^{n+1} \\
\vdots \\
\vdots \\
\phi_\mathrm{hi-1}^{n+1} \\
\phi_\mathrm{hi}^{n+1} \\
\end{array}
\right )
=
\left (
\begin{array}{c}
\phi_\mathrm{lo}^{n} \\
\phi_\mathrm{lo+1}^{n} \\
\phi_\mathrm{lo+2}^{n} \\
\vdots \\
\vdots \\
\phi_\mathrm{hi-1}^{n} \\
\phi_\mathrm{hi}^{n} \\
\end{array}
\right )
$$

We note a few things:

* This is a [tridiagonal matrix](https://en.wikipedia.org/wiki/Tridiagonal_matrix).
  Efficient solution methods exist for this that take advantage of the [sparsity pattern](https://en.wikipedia.org/wiki/Sparse_matrix).
  
* The ghost cells in our grid do not explicitly appear in the system

* The matrix is [diagonally dominant](https://en.wikipedia.org/wiki/Diagonally_dominant_matrix).  This means that
  it will work well with iterative methods.

# Solving a Tridiagonal System

We want to solve a tridiagonal system of the form:
    
$$
\left ( \begin{array}{ccccccc}
   b_0 & c_0 \\
   a_1 & b_1 & c_1 \\
       & a_2 & b_2 & c_2 \\
       &     & \ddots & \ddots & \ddots \\
       &     &        & \ddots & \ddots & \ddots \\
       &     &        &        & a_{N-2} & b_{N-2} & c_{N-2} \\
       &     &        &        &         & a_{N-1} & b_{N-1} \\
\end{array}
\right )
\left (
\begin{array}{c} x_0 \\ x_1 \\ x_2 \\ \vdots \\ \vdots \\ x_{N-2} \\ x_{N-1} \\\end{array} \right )
  =
  \left ( \begin{array}{c} d_0 \\ d_1 \\ d_2 \\ \vdots \\ \vdots \\ d_{N-2} \\ d_{N-1} \\\end{array} \right )
$$

Here, the main diagonal is represented by $b_i$, which runs from $i =
0, \ldots, N-1$.  The two off diagonals have one fewer entries.

We can solve this just using Gaussian elimination, as before, but we
won't need to pivot, since the structure is the same on all rows.  The
result is the [tridiagonal matrix
solver](https://en.wikipedia.org/wiki/Tridiagonal_matrix_algorithm).
Notice that we only need to update 2 elements in each row instead of
$N$, so by taking advantage of this pattern, this solver will be much
faster than the full Gaussian elimination.


## Solving in python

In python, we can use [`linalg.solve_banded`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.linalg.solve_banded.html).  This works
by packed the diagonals into a 2D array, as shown on the pages there.

# Direct Solve for Implicit Diffusion

We'll solve the linear system resulting from our implicit discretization
directly:

$$
\left (
\begin{array}{ccccccc}
1+\alpha &   -\alpha &           &        &         &           &          \\
-\alpha  & 1+2\alpha & -\alpha   &        &         &           &          \\
         & -\alpha   & 1+2\alpha & -\alpha&         &           &          \\
         &           & \ddots    & \ddots & \ddots  &           &          \\
         &           &           & \ddots & \ddots  & \ddots    &          \\
         &           &           &        & -\alpha & 1+2\alpha &-\alpha   \\
         &           &           &        &         & -\alpha   &1+\alpha \\
\end{array}
\right )
\left (
\begin{array}{c}
\phi_\mathrm{lo}^{n+1} \\
\phi_\mathrm{lo+1}^{n+1} \\
\phi_\mathrm{lo+2}^{n+1} \\
\vdots \\
\vdots \\
\phi_\mathrm{hi-1}^{n+1} \\
\phi_\mathrm{hi}^{n+1} \\
\end{array}
\right )
=
\left (
\begin{array}{c}
\phi_\mathrm{lo}^{n} \\
\phi_\mathrm{lo+1}^{n} \\
\phi_\mathrm{lo+2}^{n} \\
\vdots \\
\vdots \\
\phi_\mathrm{hi-1}^{n} \\
\phi_\mathrm{hi}^{n} \\
\end{array}
\right )
$$

we'll use the SciPy banded solver to do the linear algebra. 

We'll use the same grid class and initial conditions as used with the explicit method.

## Implementation

Now, we have a few options for the linear algebra.  We can:

* Keep the matrix as sparse and only store the 3 diagonals that have nonzero elements.
  We can then use the SciPy [``solve_banded``](https://docs.scipy.org/doc/scipy/reference/generated/scipy.linalg.solve_banded.html) method to solve this.
  
* Construct the full dense matrix, including all the zeros.  Here, the SciPy
  [``sparse.diags``](https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.diags.html) method can
  be helpful.


We'll write a function to just do a single implicit solve of the system, with the intent that this will be called from the time-evolution loop.

From the structure of our matrix, we have a main diagonal, $d$, and an upper ($u$) and lower ($l$) diagonal.  These have the form:

$${\bf u} = (0, -\alpha, -\alpha, \ldots, -\alpha, -\alpha)$$

$${\bf d} = (1+\alpha, 1+2\alpha, 1+2\alpha, \ldots, 1+2\alpha, 1+\alpha)$$

$${\bf l} = (-\alpha, -\alpha, -\alpha, \ldots, -\alpha, 0)$$

Where we keep $l$ and $u$ both to be length $N$, but insert a $0$ for the column where they are not defined.

For SciPy, we'll make a matrix that has 3 rows:

$${\bf A} = \left ( \begin{array}{c} {\bf u} \\ {\bf d} \\ {\bf l} \\ \end{array} \right )$$

This will be a $3\times N$ matrix, involved in the equation
$$
{\bf A} \cdot {\bf \phi}^{n+1} = {\bf \phi}^n
$$
Writing explicitly in terms of indices,
$$
A_{ij} \cdot \phi_j^{n+1} = \phi_i^n.
$$

In [ ]:
def implicit_step(gr, phi, k, dt):
    """ diffuse phi implicitly through timestep dt """

    phinew = gr.scratch_array()

    alpha = k * dt / gr.dx**2

    # create the RHS of the matrix
    R = phi[gr.ilo:gr.ihi+1]

    # create the diagonal, d+1 and d-1 parts of the matrix
    d = (1.0 + 2.0*alpha)*np.ones(gr.nx)

    u = -alpha*np.ones(gr.nx)
    u[0] = 0.0

    l = -alpha*np.ones(gr.nx)
    l[-1] = 0.0

    # set the boundary conditions by changing the matrix elements

    # homogeneous neumann
    d[0] = 1.0 + alpha
    d[-1] = 1.0 + alpha

    # solve
    A = np.matrix([u, d, l])

    phinew[gr.ilo:gr.ihi+1] = linalg.solve_banded((1, 1), A, R)

    return phinew

Here's the main evolution loop.  It works in the same fashion as our explicit implementation.

In [ ]:
def diffuse_implicit(nx, k, C, tmax, init_cond):
    """
    the main evolution loop.  Evolve

     phi_t = k phi_{xx}

    from t = 0 to tmax
    """

    # create the grid
    ng = 0
    
    g = pdemod.Grid(nx, ng)

    # time info
    dt = C * 0.5 *g.dx**2 / k
    t = 0.0

    # initialize the data
    g.phi[:] = init_cond(g, k)

    while t < tmax:

        # make sure we end right at tmax
        if t + dt > tmax:
            dt = tmax - t

        # diffuse for dt
        phinew = implicit_step(g, g.phi, k, dt)

        g.phi[:] = phinew[:]
        t += dt

    return g

## Trends with timestep size

Now we can test it out.  Implicit methods are stable for all timesteps, but keep in mind that stability is not the same as accuracy.  Let's vary the timestep size to see what happens.

In [ ]:
nx = 128
k = 1
t_diffuse = (1.0/nx)**2 / k

fig, ax = plt.subplots()

tmax = 10 * t_diffuse

for C in [0.8, 2, 10]:
    g = diffuse_implicit(nx, k, C, tmax, gaussian_ic)
    ax.plot(g.x[g.ilo:g.ihi+1], g.phi[g.ilo:g.ihi+1], label=f"C = {C}")

ax.set_xlim(0.35, 0.65)
ax.set_xlabel("$x$")
ax.set_ylabel(r"$\phi$")

ax.set_title(f"t = {tmax:7.3g}")
ax.legend()
plt.show()

We see that even at a large value of $C$, the solution remains stable.

## Convergence

Let's look at the convergence of this discretization

In [ ]:
nx_reference = 64
t_diffuse = (1.0/nx_reference)**2 / k

fig, ax = plt.subplots()

Ns = [16, 32, 64, 128, 256, 512]
dxs = []
errors = []

tmax = 10 * t_diffuse
C = 0.8

for nx in Ns:
    g = diffuse_implicit(nx, k, C, tmax, gaussian_ic)
    phi_analytic = gaussian_ic(g, k, t=tmax)
    dxs.append(g.dx)
    errors.append(g.norm(g.phi - phi_analytic))
    
ax.scatter(dxs, errors, marker="x")

ideal = errors[-1] * np.array(dxs)**2 / dxs[-1]**2

ax.plot(dxs, ideal, color="0.5", ls=":", label=r"$\mathcal{O}(\Delta x^2)$")

ax.legend()

ax.set_xscale("log")
ax.set_yscale("log")

ax.set_xlabel(r"$\Delta x$")
ax.set_ylabel("absolute error")

plt.show()

We see second-order convergence, as with the explicit method.  

**Note:**
The errors are _slightly_ worse
than the explicit discretion in general, but we do much better at coarse resolutions.

# Exercises (10 pt)

**NOTE:** An extra 2 pts are added if completed exercises are submitted before the end of class.

## Exercise 1 - Muddiest Moment

What was the most confusing part of lecture?

**Answer here**


## Exercise 2: Crank-Nicolson method (8 pt)

The [Crank-Nicolson](https://en.wikipedia.org/wiki/Crank%E2%80%93Nicolson_method) method uses a time-centered
discretization of the diffusion term:

$$\frac{\phi_i^{n+1} - \phi_i^n}{\Delta t} = \frac{k}{2}
         \left (\frac{\phi_{i+1}^n - 2\phi_i^n + \phi_{i-1}^n}{\Delta x^2} +
                \frac{\phi_{i+1}^{n+1} - 2\phi_i^{n+1} + \phi_{i-1}^{n+1}}{\Delta x^2}\right )$$
                
This is still an implicit update, but the coefficients and righthand side
in the linear system will be different

### Exercise 2(a) - Explicit Discritization (2 pt)

If we define

$$\alpha = \frac{k\Delta t}{\Delta x^2}$$

as before, then we can write this as:

$$
- \frac{\alpha}{2} \phi^{n+1}_{i-1} + (1 + \alpha) \phi_i^{n+1} - \frac{\alpha}{2} \phi_{i+1}^{n+1} = - \frac{\alpha}{2} \phi^{n}_{i-1} + (1 + \alpha) \phi_i^{n} - \frac{\alpha}{2} \phi_{i+1}^{n}
$$

This is a coupled set of algebraic equations for the new time state of $\phi$&mdash;a linear system.

for Neumann boundary conditions, we want the
  gradient at the boundary to be zero.  This means:
  
  $$\frac{\phi_\mathrm{lo} - \phi_{\mathrm{lo}-1}}{\Delta x} = 0$$
  
  This is second-order accurate at the physical boundary, $x_{\mathrm{lo}-1/2}$. This equation can be re-arranged to
  $$
  \phi_\mathrm{lo} = \phi_{\mathrm{lo} - 1}.
  $$
  
  Inserting this into our implicit system above, we find that the update for
  the first zone inside the boundary is:
  
  $$\left(1 + \frac{\alpha}{2}\right) \phi_\mathrm{lo}^{n+1} - \frac{\alpha}{2} \phi_{\mathrm{lo}+1}^{n+1} = \left(1 + \frac{\alpha}{2}\right) \phi_\mathrm{lo}^{n} - \frac{\alpha}{2} \phi_{\mathrm{lo}+1}^{n}$$

Our matrix that does the main implicit solve has has the form again

$${\bf A} = \left ( \begin{array}{c} {\bf u} \\ {\bf d} \\ {\bf l} \\ \end{array} \right )$$

with

$${\bf u} = (0, -\alpha/2, -\alpha/2, \ldots, -\alpha/2, -\alpha/2)$$

$${\bf d} = (1+\alpha/2, 1+\alpha, 1+\alpha, \ldots, 1+\alpha, 1+\alpha/2)$$

$${\bf l} = (-\alpha/2, -\alpha/2, -\alpha/2, \ldots, -\alpha/2, 0)$$

Re-write the implicit step algorithm for the crank nicolson scheme below. Blocks that you need to modify have been explicitly commented.

In [ ]:
def implicit_step_crank_nicolson(gr, phi, k, dt):
    """ diffuse phi implicitly through timestep dt """

    phinew = gr.scratch_array()

    alpha = k * dt / gr.dx**2

    # create the RHS of the matrix
    R = # ADD CODE HERE

    # create the diagonal, d+1 and d-1 parts of the matrix
    d = # ADD CODE HERE

    u = # ADD CODE HERE
    u[0] = 0.0

    l = # ADD CODE HERE
    l[-1] = 0.0

    # set the boundary conditions by changing the matrix elements

    # homogeneous neumann
    d[0] = # ADD CODE HERE
    d[-1] = # ADD CODE HERE

    # solve
    A = np.matrix([u, d, l])

    phinew[gr.ilo:gr.ihi+1] = linalg.solve_banded((1, 1), A, R)

    return phinew

### Exercise 2(b) - Re-write implicit solver (2 pt)

Re-write the step size integrator below, to include this new algorithm you just wrote. The code should be clear what you have to modify, but might not run if the previous cell has some errors.

In [ ]:
def diffuse_implicit_crank_nicolson(nx, k, C, tmax, init_cond):
    """
    the main evolution loop.  Evolve

     phi_t = k phi_{xx}

    from t = 0 to tmax
    """

    # create the grid
    ng = 0
    
    g = pdemod.Grid(nx, ng)

    # time info
    dt = C * 0.5 *g.dx**2 / k
    t = 0.0

    # initialize the data
    g.phi[:] = # ADD CODE HERE

    while t < tmax:

        # make sure we end right at tmax
        if t + dt > tmax:
            dt = tmax - t

        # diffuse for dt
        phinew = # ADD CODE HERE

        g.phi[:] = phinew[:]
        t += dt

    return g

### Exercise 2(c) - Step size trends (2 pt)

Modify the code below to find the trends with step size for this new algorithm.

In [ ]:
nx = 128
k = 1
t_diffuse = (1.0/nx)**2 / k

fig, ax = plt.subplots()

tmax = 10 * t_diffuse

for C in [0.8, 2, 10]:
    g = ## ADD CODE HERE
    ax.plot(g.x[g.ilo:g.ihi+1], g.phi[g.ilo:g.ihi+1], label=f"C = {C}")

ax.set_xlim(0.35, 0.65)
ax.set_xlabel("$x$")
ax.set_ylabel(r"$\phi$")

ax.set_title(f"t = {tmax:7.3g}")
ax.legend()
plt.show()

### Exercise 2(d) - Convergence trends (2 pt)

Modify the code below to find the convergence trends for this algorithm

In [ ]:
nx_reference = 64
t_diffuse = (1.0/nx_reference)**2 / k

fig, ax = plt.subplots()

Ns = [16, 32, 64, 128, 256, 512]
dxs = []
errors = []

tmax = 10 * t_diffuse
C = 0.8

for nx in Ns:
    g = # ADD CODE HERE
    phi_analytic = gaussian_ic(g, k, t=tmax)
    dxs.append(g.dx)
    errors.append(g.norm(g.phi - phi_analytic))
    
ax.scatter(dxs, errors, marker="x")

ideal = errors[-1] * np.array(dxs)**2 / dxs[-1]**2

ax.plot(dxs, ideal, color="0.5", ls=":", label=r"$\mathcal{O}(\Delta x^2)$")

ax.legend()

ax.set_xscale("log")
ax.set_yscale("log")

ax.set_xlabel(r"$\Delta x$")
ax.set_ylabel("absolute error")

plt.show()